# 👑 CASIE Cybersecurity Event Extraction - V3 Ultra Edition
### รวม 4 สุดยอดเทคนิค NLP ระดับ SOTA ดัน Precision, Recall และ F1 สู่จุดสูงสุด
---
**4 ยอดเทคนิคใน V3:**
1. **Class Weighting (`o_weight = 0.35`)**: ลงโทษหนักขึ้นเมื่อทาย Entity ไซเบอร์ผิด แก้ปัญหาคำธรรมดา `O` ล้น 85%+ ช่วยดันค่า **Recall (R)** และ **F1** สูงขึ้นอย่างเห็นได้ชัด
2. **Transition Constraints / Viterbi BIO Decoder**: ดักจับและกำจัดการเปลี่ยนป้ายผิดไวยากรณ์ 1,891 รูปแบบ (เช่น `O` โดดไป `I-Victim`) รับประกัน Span ถูกต้อง 100%
3. **Dual SOTA Backbones (`SecBERT` vs `microsoft/deberta-v3-base`)**: เปรียบเทียบผู้เชี่ยวชาญไซเบอร์กับราชาวงการ NER พร้อมความยาว Sequence 256 คำ
4. **Multi-Model Ensemble Averaging**: รวมพลังโมเดล SecBERT + DeBERTa-v3 เพื่อเฉลี่ยคะแนนความน่าจะเป็น ดัน F1 เพิ่มฟรีอีก +2% ถึง +3%!

**คำแนะนำการรัน:**
1. เมนู **Runtime** -> **Change runtime type** -> เลือก **T4 GPU**
2. กดรันทีละ Cell ตามลำดับ หรือกด **Runtime -> Run all**

In [ ]:
# 1. ตรวจสอบการเชื่อมต่อการ์ดจอ (GPU)
!nvidia-smi

In [ ]:
# 2. ติดตั้งแพ็กเกจที่จำเป็นสำหรับ V3 (รวม sentencepiece สำหรับ DeBERTa-v3)
!pip install -q transformers datasets accelerate sentencepiece

In [ ]:
# 3. อัปโหลดไฟล์ colab_package_v3.zip (ขนาด 2.5 MB) จากคอมพิวเตอร์ของคุณ
from google.colab import files
import os, zipfile

if not os.path.exists('dataset'):
    print('กรุณากด Choose Files เพื่ออัปโหลด colab_package_v3.zip:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as zip_ref:
                zip_ref.extractall('.')
            print(f'แตกไฟล์ {fn} สำเร็จ!')
else:
    print('ชุดข้อมูล dataset และสคริปต์ V3 พร้อมใช้งานแล้ว!')

In [ ]:
# 4. เริ่มเทรนกลุ่ม SecBERT V3 (Model #1 Cybersecurity Specialist ครบ 4 โหมด)
print('========== [Run 1/12] SecBERT Baseline (Text Only) ==========')
!python step2_train_experiments.py --model_name jackaduma/SecBERT --feature_mode none --o_weight 0.35 --epochs 10 --batch_size 16 --do_train

print('========== [Run 2/12] SecBERT + POS Tag ==========')
!python step2_train_experiments.py --model_name jackaduma/SecBERT --feature_mode pos --o_weight 0.35 --epochs 10 --batch_size 16 --do_train

print('========== [Run 3/12] SecBERT + General NER Tag ==========')
!python step2_train_experiments.py --model_name jackaduma/SecBERT --feature_mode ner --o_weight 0.35 --epochs 10 --batch_size 16 --do_train

print('========== [Run 4/12] SecBERT + POS + General NER (Full Features) ==========')
!python step2_train_experiments.py --model_name jackaduma/SecBERT --feature_mode pos_ner --o_weight 0.35 --epochs 10 --batch_size 16 --do_train

In [ ]:
# 4.5 เริ่มเทรนกลุ่ม BERT-base V3 (Model #2 Standard Baseline ครบ 4 โหมด)
print('========== [Run 5/12] BERT-base Baseline (Text Only) ==========')
!python step2_train_experiments.py --model_name bert-base-uncased --feature_mode none --o_weight 0.35 --epochs 15 --patience 10 --batch_size 16 --do_train

print('========== [Run 6/12] BERT-base + POS Tag ==========')
!python step2_train_experiments.py --model_name bert-base-uncased --feature_mode pos --o_weight 0.35 --epochs 15 --patience 10 --batch_size 16 --do_train

print('========== [Run 7/12] BERT-base + General NER Tag ==========')
!python step2_train_experiments.py --model_name bert-base-uncased --feature_mode ner --o_weight 0.35 --epochs 15 --patience 10 --batch_size 16 --do_train

print('========== [Run 8/12] BERT-base + POS + General NER (Full Features) ==========')
!python step2_train_experiments.py --model_name bert-base-uncased --feature_mode pos_ner --o_weight 0.35 --epochs 15 --patience 10 --batch_size 16 --do_train

In [ ]:
# 5. เริ่มเทรนกลุ่ม RoBERTa-base V3 (Model #3 Meta/Facebook AI ครบทั้ง 4 โหมด)
print('========== [Run 9/12] RoBERTa-base Baseline (Text Only) ==========')
!python step2_train_experiments.py --model_name roberta-base --feature_mode none --o_weight 0.35 --epochs 15 --patience 10 --batch_size 16 --do_train

print('========== [Run 10/12] RoBERTa-base + POS Tag ==========')
!python step2_train_experiments.py --model_name roberta-base --feature_mode pos --o_weight 0.35 --epochs 15 --patience 10 --batch_size 16 --do_train

print('========== [Run 11/12] RoBERTa-base + General NER Tag ==========')
!python step2_train_experiments.py --model_name roberta-base --feature_mode ner --o_weight 0.35 --epochs 15 --patience 10 --batch_size 16 --do_train

print('========== [Run 12/12] RoBERTa-base + POS + General NER (Full Features) ==========')
!python step2_train_experiments.py --model_name roberta-base --feature_mode pos_ner --o_weight 0.35 --epochs 15 --patience 10 --batch_size 16 --do_train

In [ ]:
# 6. ตรวจข้อสอบ Test Set 100 ข่าว พร้อมประเมินผล Multi-Model Ensemble หลอมรวมพลังโมเดล!
!python step3_evaluate_experiments.py --all --ensemble

In [ ]:
# 7. แสดงผลกราฟ Loss Curve และดาวน์โหลดไฟล์ผลลัพธ์ทั้งหมดกลับลงคอมพิวเตอร์ของคุณ
from google.colab import files
from IPython.display import Image, display
import glob

print('📈 แสดงตัวอย่างกราฟ Loss Curve พร้อมจุด Early Stop (*):')
for img in sorted(glob.glob('models/*/loss_curve.png')):
    print(f'\nผลของ: {img}')
    display(Image(img, width=650))

# บีบอัดผลลัพธ์ทั้งหมด (ตารางคะแนน Master CSV, ตาราง Log แต่ละรอบ, รูปกราฟ และโมเดล)
!zip -r all_v3_results_and_graphs.zip models/
files.download('all_v3_results_and_graphs.zip')
print('🎉 ดาวน์โหลด all_v3_results_and_graphs.zip เรียบร้อยแล้ว!')